# TIB PCB Commissioning

Measure what **working** means for this assembled PCB: routes and connector identity, levels and losses,
calibrated range, noise, response, laser current granularity, and throughput behavior. Run each experiment
individually in the lab and chamber. This is an editable scientific notebook, not an automatic acceptance test.

New code lives here; `hispec_fibpcb.py` owns the command interface, collectors, and firmware-model evaluation.
The original [attenuator lab](attenuator_calibration_lab.ipynb), [throughput lab](throuput_monitor_lab.ipynb),
and [Copy1 lab](atten_and_tput_cal_and_noise_lab-Copy1.ipynb) remain intact. Electrical scope work is in
[tib_fvoa_noise_scope_lab.ipynb](tib_fvoa_noise_scope_lab.ipynb), which runs independently.

Use the workspace `.venv` kernel. Hardware switches start disabled so **Run All can be used for offline review**.
Enable only the experiment being run. Load existing NPZ files without connecting. There is no prescribed total
runtime or guaranteed warmup time; keep transients and decide later which samples are stationary.

1. Configuration, inventory, board operation, dark
2. Physical routing and assembled static losses
3. Embedded autocalibration and combined fit figures
4. Adjacent current steps and independent fixed-current holds
5. Noise, response, throughput, and room/chamber/build comparisons

Firmware references: [commands](../doc/commands.md), [hardware](../doc/hardware.md),
[calibration](../doc/attenuator_calibration.md), [PD notes](../doc/photodiode_notes.md).

In [ ]:
from pathlib import Path
from dataclasses import asdict, is_dataclass, fields
from datetime import datetime, timezone
import math
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import display

TOOLS = next(p.resolve() for p in (Path.cwd(), Path.cwd() / 'tools', Path.cwd() / 'hispec-tib/tools')
             if (p / 'hispec_fibpcb.py').is_file())
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))
import hispec_fibpcb as hspcb

plt.rcParams.update({'axes.grid': True, 'grid.alpha': 0.18, 'figure.figsize': (11, 5)})

In [ ]:
CHANNEL = 'yj'                         # Copy/change to 'hk' for the other channel.
LASERS = {'yj': ['1028y', '1270j', '1430yj'], 'hk': ['1430hk', '1510h', '2330k']}[CHANNEL]
BROKER, DEVICE = 'hispec.caltech.edu', 'hsfib-tib'
OUTPUT, FIBER = f'{CHANNEL}_ao', 'M'    # Match the physical patch before each experiment.
CONDITION = 'room'                     # e.g. 'room', 'chamber_-5C'; measured temperatures are also saved.
BUILD_LABEL = 'enter flashed build / autolevel policy'
STARTUP_HISTORY = 'enter cold bank / previously enabled / elapsed idle time'
PATCH_NOTES = 'enter actual fiber endpoints, jumpers, fixed attenuators, and meters'
FLASHED_ADC_SPS = 64                   # Owner-specified flashed setup, not queried by the current API.
DETECTOR_BANDWIDTH_HZ = {'yj': 20.0, 'hk': 500.0}[CHANNEL]
ADC_INPUT_CAPACITANCE_UF = 0.47
CONNECT = False
pcb = globals().get('pcb')

DATA_DIR = TOOLS / 'commissioning_data'
CAPTURE_FILES = globals().get('CAPTURE_FILES', [])
CALIBRATION_FILES = globals().get('CALIBRATION_FILES', {})

## Configuration and archives

Every new archive is a compressed NPZ with named arrays and scalar-column tables. `load_tables(path)` returns
DataFrames; `context` is a typed settings table (`key`, `kind`, `number`, `text`). Raw samples, raw calibration
records, fits, bridge anchors, timestamps, temperatures, conditions, and errors are retained. No new JSON or
pickle payloads. The only JSON reader below opens existing calibration archives from the earlier lab.

The installed build is not inferred from the repository: enter its label above. Current source defaults to
attenuation-first dimming while `commands.md` describes laser-first as the default; compare the two actual
flashed builds. The owner-specified 64 SPS ADC, YJ 20 Hz detector, HK 500 Hz detector, and 0.47 µF ADC-input
capacitor are recorded as setup assumptions. Capacitance alone does not specify the analog cutoff.

In [ ]:
def parameter_table(values):
    """Flatten settings into a readable, typed table; no JSON or pickled objects."""
    rows = []
    def visit(key, value):
        if is_dataclass(value):
            value = asdict(value)
        if isinstance(value, dict):
            for name, item in value.items():
                visit(f'{key}.{name}' if key else str(name), item)
        elif isinstance(value, (tuple, list, np.ndarray)):
            for i, item in enumerate(value):
                visit(f'{key}.{i}', item)
        else:
            numeric = isinstance(value, (int, float, np.number)) and not isinstance(value, (bool, np.bool_))
            rows.append((key, 'number' if numeric else 'text', float(value) if numeric else np.nan,
                         '' if numeric or value is None else str(value)))
    visit('', values)
    return pd.DataFrame(rows, columns=['key', 'kind', 'number', 'text'])


def parameter(table, key, default=np.nan):
    """Read one saved scalar without reconstructing a driver object."""
    rows = table.loc[table.key.eq(key)]
    if rows.empty:
        return default
    row = rows.iloc[-1]
    return row.number if row.kind == 'number' else row.text


def table_records(frame):
    """Keep numeric dtypes; encode text and stream flag tuples as Unicode for allow_pickle=False."""
    if not len(frame.columns):
        return np.empty(len(frame), dtype=[])
    frame = frame.copy()
    string_types = {}
    for name in frame:
        if frame[name].dtype.kind == 'O' or isinstance(frame[name].dtype, pd.StringDtype):
            frame[name] = frame[name].map(lambda v: '|'.join(v) if isinstance(v, tuple) else '' if v is None else str(v))
            string_types[name] = f'U{max(1, frame[name].str.len().max() if len(frame) else 1)}'
    return frame.to_records(index=False, column_dtypes=string_types)


def save_tables(stem, **tables):
    """Archive raw acquisition before analysis. Exclusive creation never overwrites a capture."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    path = DATA_DIR / f'{stem}_{stamp}.npz'
    arrays = {name: table_records(value) if isinstance(value, pd.DataFrame) else np.asarray(value)
              for name, value in tables.items()}
    if any(a.dtype.hasobject for a in arrays.values()):
        raise TypeError('NPZ tables must have scalar numeric/Unicode columns, not Python objects.')
    with path.open('xb') as file:
        np.savez_compressed(file, **arrays)
    print(path)
    return path


def load_tables(path):
    """Offline read of this notebook's named arrays and structured tables."""
    with np.load(path, allow_pickle=False) as archive:
        return {key: pd.DataFrame.from_records(archive[key]) if archive[key].dtype.names is not None else archive[key].copy()
                for key in archive.files}


def snapshot(client, laser, *, output=None, fiber=None):
    """Query active settings and selected-laser telemetry; never request light or an all-bank PD read."""
    output, fiber = output or OUTPUT, fiber or FIBER
    source = f'{CHANNEL}_1430' if laser.startswith('1430') else f'{CHANNEL}_laser'
    values = dict(utc=datetime.now(timezone.utc).isoformat(), laser=laser, channel=CHANNEL,
        output=output, fiber=fiber, condition=CONDITION, build_label=BUILD_LABEL,
        startup_history=STARTUP_HISTORY, patch_notes=PATCH_NOTES, flashed_adc_sps=FLASHED_ADC_SPS,
        detector_bandwidth_hz=DETECTOR_BANDWIDTH_HZ, adc_input_capacitance_uf=ADC_INPUT_CAPACITANCE_UF,
        status=client.status(), clock=client.time(), heater=client.laser_bankheater(),
        bank=client.laser_bankpower(), laser_status=client.laser(laser), engineering=client.laser_status(laser),
        laser_settings=client.laser_settings(laser), coeff=client.atten_coeff(laser), atten=client.atten(laser),
        pd_settings=client.pd_settings(CHANNEL), dark=client.pd_dark(CHANNEL), switches=client.mems(),
        launch_loss=client.mems_route_loss(f'{source}_to_{output}'),
        return_loss=client.mems_route_loss(f'{CHANNEL}_{"mm" if fiber == "M" else "sm"}_to_{CHANNEL}_pd'))
    return parameter_table(values)

In [ ]:
if CONNECT:
    if pcb is None:
        pcb = hspcb.HispecFibPcb(BROKER, device=DEVICE, connect=True, auto_connect=False)
    elif not pcb.is_connected:
        pcb.connect()
    inventory = pd.concat([snapshot(pcb, name).assign(source_laser=name) for name in LASERS], ignore_index=True)
    display(inventory)
    inventory_file = save_tables('inventory', inventory=inventory)
else:
    print('Offline: definitions and archived-data analysis are available.')

## Board operation and dark

The selected channel covers three laser/attenuator pairs and four MEMS switches; run the HK copy for the other
half of the PCB. Inventory records identities, expected serials, relay errors, communications/time, switch intent,
and active settings. Static MEMS commands save switch intent automatically. A successful MEMS reply reports the last commanded state, not optical verification.

Bank/heater/PD relay operations below are separate experiments. Heater mode commands persist their policy
automatically; finish with the desired mode. Bank `override_off` stops all lasers and
changes thermal history. `pd(channel)` powers/refreshes that detector; use it deliberately. Heater operation
is observed with measured temperatures, especially at −5 °C. Firmware monitoring, networking, watchdog and
fault handling are not exercised by intentional fault injection here.

In [ ]:
RUN_POWER_EXERCISE = False
POWER_DWELL_S = 2.0                     # Change for thermal observation; not a settling guarantee.
POWER_STEPS = [('bank', 'override_on'), ('heater', 'override_on'), ('heater', 'override_off'),
               ('heater', 'auto'), ('pd', 'override_off'), ('pd', 'override_on'),
               ('pd', 'auto'), ('bank', 'override_off'), ('bank', 'auto')]
if RUN_POWER_EXERCISE:
    pcb.stop_throughput('all')
    power_rows = []
    try:
        for component, mode in POWER_STEPS:
            begin = time.time_ns() // 1_000_000
            if component == 'bank':
                reply = pcb.laser_bankpower(mode)
            elif component == 'heater':
                reply = pcb.laser_bankheater(mode)
            else:
                reply = pcb.pd_settings(CHANNEL, power=mode, persist=False)
            time.sleep(POWER_DWELL_S)
            power_rows.append(parameter_table(dict(component=component, mode=mode, host_ms=begin,
                reply=reply, status=pcb.status(), temperatures=pcb.temps(), heater=pcb.laser_bankheater())))
    finally:
        power_file = save_tables('power', observations=pd.concat(power_rows, keys=range(len(power_rows)))
                                .reset_index(level=0).rename(columns={'level_0': 'step'}) if power_rows
                                else parameter_table({}))
    display(load_tables(power_file)['observations'])

In [ ]:
def open_stream(client, seconds, *, laser='none', fiber=None, output=None, autolevel=False, initial_level=None):
    """Stop an earlier owner BEFORE manual source setup; return a new finite-capacity collector.

    A former autolevel owner can shut down its source here. The returned manual/passive
    collector has no inherited shutdown obligation. Its stop leaves manually enabled lasers alone.
    """
    if not np.isfinite(seconds) or seconds <= 0:
        raise ValueError('Choose a positive capture duration.')
    client.stop_throughput(CHANNEL)
    return client.measure_throughput(laser, channel=CHANNEL, fiber=fiber or FIBER,
        output=(output or OUTPUT) if laser != 'none' else None, autolevel=autolevel,
        initial_level=initial_level, off_in_s=0, collect=True, format='binary',
        max_samples=max(1000, math.ceil(seconds / .05 * 1.5) + 1000))


def collect_trace(client, monitor, laser, seconds, label, *, context, extra=None, telemetry_s=2.0):
    """Block for an editable hold; save on completion or interrupt and detach the stream.

    Slow engineering queries give measured current/TEC context, not high-frequency current noise.
    Raw samples include the start transient; settling cuts belong in analysis. No automatic laser STOP.
    """
    started = time.monotonic()
    readings, error, completed = [], '', False
    try:
        next_read = started
        while time.monotonic() - started < seconds:
            if time.monotonic() >= next_read:
                before = time.time_ns() // 1_000_000
                status = client.laser_status(laser)
                ambient = client.status().amb_c
                heater = client.laser_bankheater()
                after = time.time_ns() // 1_000_000
                # PID configuration is already in the context table; each row keeps scalar telemetry.
                values = {k: v for k, v in asdict(status).items() if k != 'pid'}
                readings.append(dict(host_start_ms=before, host_end_ms=after, ambient_c=ambient,
                    heater_on=heater.heater_on, heater_auto_state=heater.auto_state, **values))
                next_read = time.monotonic() + telemetry_s
            time.sleep(min(.1, max(0, seconds - (time.monotonic() - started))))
        completed = True
    except BaseException as exc:
        error = f'{type(exc).__name__}: {exc}'
        raise
    finally:
        # Stop/detach first to include queued telemetry. Save even if the stop command fails.
        stop_error = ''
        try:
            monitor.stop()
        except Exception as exc:
            stop_error = f'{type(exc).__name__}: {exc}'
            raise
        finally:
            frame = monitor.to_dataframe()
            for name, value in dict(experiment=label, source_laser=laser, condition=CONDITION,
                                    build_label=BUILD_LABEL, **(extra or {})).items():
                frame[name] = value
            telemetry = pd.DataFrame(readings)
            outcome = parameter_table(dict(complete=completed, error=error, stop_error=stop_error, requested_s=seconds,
                elapsed_s=time.monotonic()-started, samples=len(frame), collector_capacity=monitor.max_samples,
                capacity_reached=len(frame) >= monitor.max_samples))
            path = save_tables(f'{label}_{laser}', samples=frame, telemetry=telemetry, context=context, outcome=outcome)
            CAPTURE_FILES.append(path)
    return path

In [ ]:
RUN_DARK = False
DARK_DURATION_MS = 2000
DARK_TRACE_SECONDS = 20.0
if RUN_DARK:
    pcb.stop_throughput('all')
    for name in LASERS:
        pcb.laser(name, value=0, autooff_s=0)
    # Confirm all external/manual inputs to this PD are dark as well.
    pcb.pd(CHANNEL)
    dark = pcb.pd_dark(CHANNEL, duration_ms=DARK_DURATION_MS, persist=False)
    while dark.pending:
        time.sleep(.25)
        dark = pcb.pd_dark(CHANNEL)
    display(dark)
    dark_file = save_tables('dark_settings', dark=parameter_table(asdict(dark)))
    context = snapshot(pcb, LASERS[0])
    monitor = open_stream(pcb, DARK_TRACE_SECONDS)
    collect_trace(pcb, monitor, LASERS[0], DARK_TRACE_SECONDS, 'dark', context=context)

ACCEPT_DARK = False                    # Run after reviewing the captured mean/RMS.
if ACCEPT_DARK:
    dark = pcb.pd_dark(CHANNEL)
    if dark.pending:
        raise RuntimeError('Wait for dark capture to finish.')
    pcb.pd_dark(CHANNEL, dark_mv=dark.dark.mean_mv, rms_mv=dark.dark.rms_mv, persist=True)

## Physical routes and connector identity

Manually patch one physical output to one return with a known-loss jumper. Record that physical patch before
measuring intended and alternate states; do not move it while running a group. The editable table covers each
laser at AO/FEI and both return fibers, external calibration input, and **all four 1430 retro splitter outputs**.
Run one patch group, repatch, then run the next. Use repeated A→B→A selection to see repeatability and settling.
No endpoint identity can be established from the switch software state alone.

For retro, the `forward_retro` switch is driven directly to B. Return-only `laser='none'` streaming deliberately
leaves launch routing alone. `mems_split` is an AS-board command and does not control the TIB passive splitter.
Record retro output launch powers with the external meter before bridging each output. External calibration
light is supplied manually; the notebook zeros the bank lasers in that experiment.

In [ ]:
route_rows = []
for name in LASERS:
    source = f'{CHANNEL}_1430' if name.startswith('1430') else f'{CHANNEL}_laser'
    for endpoint in ('ao', 'fei'):
        for fiber in ('M', 'S'):
            route_rows.append(dict(patch_id=f'{name}_{endpoint}_{fiber}', laser=name, source=source,
                endpoint=endpoint, fiber=fiber, kind='forward', level=.05, dac1_mv=3300., dac2_mv=3300.,
                jumper_loss_db=np.nan, extra_loss_db=0., launch_power_nw=np.nan, duration_s=5., repeats=2))
for endpoint in ('ao', 'fei'):
    for fiber in ('M', 'S'):
        route_rows.append(dict(patch_id=f'cal_{endpoint}_{fiber}', laser='external', source=f'{CHANNEL}_cal',
            endpoint=endpoint, fiber=fiber, kind='cal', level=0., dac1_mv=3300., dac2_mv=3300.,
            jumper_loss_db=np.nan, extra_loss_db=0., launch_power_nw=np.nan, duration_s=5., repeats=2))
for port in range(1, 5):
    route_rows.append(dict(patch_id=f'retro_{port}', laser=f'1430{CHANNEL}', source='retro',
        endpoint=f'retro_{port}', fiber='S', kind='retro', level=.05, dac1_mv=3300., dac2_mv=3300.,
        jumper_loss_db=np.nan, extra_loss_db=0., launch_power_nw=np.nan, duration_s=5., repeats=2))
route_plan = pd.DataFrame(route_rows).set_index('patch_id')
# Enter proven visible-light settings before acquisition; 3300/3300 is only the initial parked state.
# Example: route_plan.loc['1028y_ao_M', ['dac1_mv', 'dac2_mv']] = [2500., 2500.]
display(route_plan)

In [ ]:
RUN_ROUTE_GROUP = False
PATCH_ID = '1028y_ao_M' if CHANNEL == 'yj' else '1430hk_ao_M'
if RUN_ROUTE_GROUP:
    row = route_plan.loc[PATCH_ID]
    source_laser = LASERS[0] if row.laser == 'external' else row.laser
    states = [('intended', row.fiber, False), ('alternate_return', 'S' if row.fiber == 'M' else 'M', False),
              ('alternate_launch', row.fiber, True), ('restored', row.fiber, False)]
    for repeat in range(int(row.repeats)):
        for label, selected_fiber, alternate in states:
            # Stop old ownership before touching source current. Passive start changes only return selection.
            monitor = open_stream(pcb, row.duration_s, fiber=selected_fiber)
            try:
                for name in LASERS:
                    pcb.laser(name, value=0, autooff_s=0)
                if row.kind == 'retro':
                    pcb.mems_switch(f'{CHANNEL}_forward_retro', state='A' if alternate else 'B')
                else:
                    endpoint = ('fei' if row.endpoint == 'ao' else 'ao') if alternate else row.endpoint
                    pcb.mems_route(row.source, f'{CHANNEL}_{endpoint}')
                    if row.laser.startswith('1430'):
                        pcb.mems_switch(f'{CHANNEL}_forward_retro', state='A')
                if row.laser != 'external':
                    pcb.atten(row.laser, value1_mv=row.dac1_mv, value2_mv=row.dac2_mv)
                    pcb.laser(row.laser, value=row.level, autooff_s=0)
                context = pd.concat([snapshot(pcb, source_laser, fiber=selected_fiber, output=OUTPUT if row.kind == 'retro' else f'{CHANNEL}_{endpoint}'), parameter_table({'patch': dict(row), 'patch_id': PATCH_ID})])
                collect_trace(pcb, monitor, source_laser, row.duration_s, 'route', context=context,
                    extra=dict(patch_id=PATCH_ID, selection=label, repeat=repeat, selected_fiber=selected_fiber))
            finally:
                monitor.stop()
                if row.laser != 'external':
                    pcb.laser(row.laser, value=0, autooff_s=0)

## Assembled static losses

Transcribe the installed switch serials and directional measurements from your workbook into the table below.
Optical ports **a/b/c** and control states **A/B** are different concepts. These component measurements are
reference sums; the result being calibrated is the total assembled path, including connectors and static attenuation.

`total_loss_db = 10 log10(P_before / P_after)`. Define meter planes explicitly. If using the PD, infer detector
power from **signed net voltage / (responsivity × effective gain)**. Do not divide by the firmware return
transmission again. Separate a known jumper and deliberately added attenuation from the assembly result.
If the measurement traverses both launch and return, it only determines their combined loss unless one is
independently known. Laser power estimates and wavelength response assumptions remain visible uncertainties.

In [ ]:
SWITCH_WORKBOOK = Path('/Users/jibailey/Library/CloudStorage/OneDrive-CaliforniaInstituteofTechnology/HISPEC/HISPEC - Subsystems [L3]/Fiber Delivery Subsystem [FIB]/MEMS Switches and Splitters/MEMS_switch_testing/MEMS_switch_test_document.xlsx')
switch_losses = pd.DataFrame([
    dict(location=f'{CHANNEL}_{location}', serial='', direction=direction, loss_db=np.nan, wavelength_nm=np.nan)
    for location in ('forward_retro', 'laser_cal', 'ao_fei', 'mm_sm')
    for direction in ('a->c', 'a->b', 'b->a', 'c->a')])
# One row per component traversed; repeat switch identity/direction for each assembled path reference.
path_components = pd.DataFrame(columns=['path_id', 'component', 'serial', 'direction', 'loss_db'])
loss_measurements = pd.DataFrame(columns=['path_id', 'laser', 'before_plane', 'after_plane', 'before_nw',
    'after_nw', 'jumper_loss_db', 'extra_loss_db', 'dynamic_fvoa_db', 'known_other_path_db', 'notes'])
# dynamic_fvoa_db: subtract only if the measured assembly traverses the FVOAs at nonzero relative attenuation.
# known_other_path_db: e.g. independently measured return loss when solving launch loss from a loopback.
# Fill 0 explicitly when a correction does not apply. Missing values stay missing, never assumed lossless.
display(switch_losses)

In [ ]:
ROUTE_FILES = []                       # Explicit paths, or list(DATA_DIR.glob('route_*.npz')).
ROUTE_SETTLE_S = 1.0                   # Analysis cut only; initial samples remain in NPZ.
route_summary_rows = []
for path in ROUTE_FILES:
    saved = load_tables(path)
    f, ctx = saved['samples'], saved['context']
    if f.empty:
        continue
    f = f.loc[(f.t_ms-f.t_ms.iloc[0])/1000 >= ROUTE_SETTLE_S]
    usable = f.loc[np.isfinite(f.pd_net_mv) & (f.pd_mv < hspcb.PD_ADC_USABLE_MV)]
    r = parameter(ctx, 'pd_settings.responsivity_a_per_w')
    g = parameter(ctx, 'pd_settings.transimpedance_v_per_a')
    route_summary_rows.append(dict(file=str(path), patch_id=parameter(ctx, 'patch_id'),
        selection=saved['samples'].selection.iloc[0], repeat=saved['samples'].repeat.iloc[0],
        count=len(usable), mean_mv=usable.pd_net_mv.mean(), rms_mv=usable.pd_net_mv.std(),
        overrange=int((f.pd_mv >= hspcb.PD_ADC_USABLE_MV).sum()),
        detector_power_nw=usable.pd_net_mv.mean()*1e6/(r*g)))
route_summary = pd.DataFrame(route_summary_rows)
if not route_summary.empty:
    display(route_summary)
    display(route_summary.pivot_table(index='patch_id', columns='selection', values='mean_mv', aggfunc='mean'))
    # This is a physical-endpoint matrix, not a pass/fail claim based on switch command acknowledgments.
    route_summary.pivot_table(index='patch_id', columns='selection', values='mean_mv').plot.bar(ylabel='Signed PD net (mV)')

if not loss_measurements.empty:
    loss_results = loss_measurements.copy()
    good = (loss_results.before_nw > 0) & (loss_results.after_nw > 0)
    loss_results['measured_total_db'] = np.nan
    loss_results.loc[good, 'measured_total_db'] = 10*np.log10(loss_results.loc[good, 'before_nw']/loss_results.loc[good, 'after_nw'])
    loss_results['assembly_db'] = loss_results.measured_total_db - loss_results[
        ['jumper_loss_db', 'extra_loss_db', 'dynamic_fvoa_db', 'known_other_path_db']].sum(axis=1, min_count=4)
    references = path_components.groupby('path_id').loss_db.agg(lambda x: x.sum(min_count=len(x)))
    loss_results['reference_db'] = loss_results.path_id.map(references)
    loss_results['difference_db'] = loss_results.assembly_db - loss_results.reference_db
    display(loss_results)

In [ ]:
# Enter reviewed assembled totals, not incremental corrections to firmware defaults.
route_updates = pd.DataFrame(columns=['route', 'laser', 'loss_db'])
APPLY_ROUTE_LOSSES_RAM = False
PERSIST_ROUTE_LOSSES = False
if APPLY_ROUTE_LOSSES_RAM or PERSIST_ROUTE_LOSSES:
    pcb.stop_throughput(CHANNEL)        # A restart is required: stream route factors are latched at start.
    for row in route_updates.itertuples(index=False):
        if not np.isfinite(row.loss_db) or row.loss_db < 0:
            raise ValueError('Supply a nonnegative total path loss in dB.')
        pcb.mems_route_loss(row.route, laser=row.laser, loss=f'{row.loss_db:.12g} dB', persist=PERSIST_ROUTE_LOSSES)
    save_tables('route_loss_update', applied=route_updates, switches=switch_losses, components=path_components,
                measurements=loss_measurements, context=snapshot(pcb, LASERS[0]))
# Numeric loss arguments mean fraction LOST; stream *_route_tx fields are transmission.
# Prefer dB strings for large static attenuation. Relative FVOA fits do not include the static loss.

## Embedded autocalibration

Patch the configured output to the selected return and obtain a valid dark first. Each run exercises both physical
FVOAs over the DAC range; firmware may use full nominal laser output. It owns acquisition, bridging, fitting,
and installation of accepted coefficients in RAM. `persist=False` does **not** prevent the RAM update.

Run all three lasers or select a subset below. Each pair is fetched and archived before starting another.
Completed acquisitions with rejected fits are still scientific data. Interrupts attempt to fetch the currently
retained data before cancellation; that partial snapshot can be incomplete. Firmware cancellation clears its
records. A retrieval error is recorded and re-raised, never presented as recovered data.

In [ ]:
def save_calibration(dataset, context, laser, *, error=''):
    """Store the firmware result and raw anchors as tables before any cleanup can discard them."""
    fits = next((item['fits'] for item in dataset.meta if 'fits' in item), {})
    fit_rows, physical_rows, bridge_rows = [], [], []
    for physical in ('dac1', 'dac2'):
        fit = fits.get(physical, hspcb.AttenuatorFitMetrics(valid=False))
        row = {k: np.nan if v is None else v for k, v in asdict(fit).items() if k != 'correction_coeff'}
        row.update(physical=physical, **{f'correction_{i}': x for i, x in enumerate(fit.correction_coeff or (0.,)*6)})
        fit_rows.append(row)
        meta = dataset._physical_meta(physical)
        if meta is not None:
            physical_rows.append({k: v for k, v in meta.items() if k != 'bridges'})
            bridge_rows.extend(dict(physical=physical, **bridge) for bridge in meta.get('bridges', ()))
    status = next((item['status'] for item in dataset.meta if 'status' in item), {})
    error = error or next((item.get('retrieval_error', '') for item in dataset.meta if 'retrieval_error' in item), '')
    return save_tables(f'cal_{laser}', records=dataset.records, fits=pd.DataFrame(fit_rows),
        physical=pd.DataFrame(physical_rows, columns=list(physical_rows[0]) if physical_rows else ['physical']),
        bridges=pd.DataFrame(bridge_rows, columns=['physical', 'bridge_index', 'before_record', 'after_record']),
        status=parameter_table({'status': status, 'retrieval_error': error}), context=context)


def load_calibration(path):
    """Read new table archives or the actual older NPZ captures; no hardware connection."""
    with np.load(path, allow_pickle=False) as z:
        records = z['records'].copy()
        if 'metadata' in z.files:
            import json                   # Read-only support for existing Copy1 captures.
            saved = json.loads(str(z['metadata']))
            for item in saved['meta']:
                if 'fits' in item:
                    item['fits'] = {name: hspcb.AttenuatorFitMetrics(**value) for name, value in item['fits'].items()}
            return hspcb.AttenuatorCalibrationDataset(records, tuple(saved['meta'])), parameter_table(saved['context'])
        tables = {key: pd.DataFrame.from_records(z[key]) for key in ('fits', 'physical', 'bridges', 'context', 'status')}
    fits = {}
    for row in tables['fits'].to_dict('records'):
        physical = row['physical']
        values = {f.name: row[f.name] for f in fields(hspcb.AttenuatorFitMetrics) if f.name != 'correction_coeff'}
        values = {k: None if isinstance(v, float) and np.isnan(v) else v for k, v in values.items()}
        values['correction_coeff'] = tuple(row[f'correction_{i}'] for i in range(6))
        fits[physical] = hspcb.AttenuatorFitMetrics(**values)
    status = {f.name: parameter(tables['status'], f'status.{f.name}', None)
              for f in fields(hspcb.AttenuatorCalibrationStatus) if f.name not in ('dac1', 'dac2')}
    if status['state'] is not None:
        for key in ('n', 't_ms', 'complete_pct', 'error'):
            status[key] = int(status[key])
        status = hspcb.AttenuatorCalibrationStatus(**status, dac1=fits['dac1'], dac2=fits['dac2'])
    else:
        status = {}
    meta = [{'fits': fits, 'status': status, 'retrieval_error': parameter(tables['status'], 'retrieval_error', '')}]
    for row in tables['physical'].to_dict('records'):
        row['bridges'] = tables['bridges'].loc[tables['bridges'].physical.eq(row['physical'])].drop(columns='physical').to_dict('records')
        meta.append(row)
    return hspcb.AttenuatorCalibrationDataset(records, tuple(meta)), tables['context']


def captured_coefficients(dataset, context):
    """Use each captured fit with its acquisition gain, never the presently installed coefficients."""
    result = {}
    for physical in ('dac1', 'dac2'):
        coeff = dataset._fit_coeff_for_physical(physical)
        if coeff is not None:
            gain = parameter(context, f'coeff.{physical}.gain', hspcb.ATTENUATOR_DEFAULT_GAIN)
            result[physical] = (*coeff[:3], float(gain), coeff[4], coeff[5])
    return result

In [ ]:
RUN_AUTOCAL = False
CAL_LASERS = list(LASERS)
CAL_DWELL_MS = 550
if RUN_AUTOCAL:
    pcb.stop_throughput('all')
    for name in LASERS:
        pcb.laser(name, value=0, autooff_s=0)
    for name in CAL_LASERS:
        context = snapshot(pcb, name)
        finished, archived, start_confirmed = False, False, False
        try:
            pcb.pd(CHANNEL)              # Explicitly power/refresh PD; use the already measured dark.
            state = pcb.atten_calibrate(name, output=OUTPUT, fiber=FIBER, dwell_ms=CAL_DWELL_MS, persist=False)
            start_confirmed = True
            while state.state == 'running':
                print(f'{name} {state.physical}: {state.complete_pct}%  {state.point}', end='\r')
                time.sleep(2)
                state = pcb.atten_calibrate()
            finished = True
            dataset = pcb.atten_calibration_data(physical='all')
            path = save_calibration(dataset, context, name)
            archived = True
            CALIBRATION_FILES[name] = path
            display(state)
        except BaseException as exc:
            if not archived:
                try:
                    dataset = pcb.atten_calibration_data(physical='all')
                    path = save_calibration(dataset, context, name, error=f'Partial retrieval; start confirmed={start_confirmed}. Requested source={name}; unconfirmed starts may retain the preceding acquisition. {type(exc).__name__}: {exc}')
                    CALIBRATION_FILES[name] = path
                except Exception as retrieval_error:
                    save_tables(f'cal_error_{name}', context=context,
                        error=parameter_table(dict(acquisition_error=str(exc), retrieval_error=str(retrieval_error), data_available=False)))
            raise
        finally:
            try:
                if not finished:
                    pcb.atten_calibrate_stop()
            finally:
                pcb.laser(name, stop=True)

## Combined calibration figure and offline replay

The upper panels preserve raw dark-subtracted and bridge-scaled data. The fit overlays measured sweep records,
with a quarter-height residual panel attached directly below it. Faint separators identify segments; the
vertical line labels the DAC voltage crossing into the rough region. The tail shading starts at excluded tail
measurements, separately from the calibrated-limit line.

**Excluded** includes unsaturated sweep points omitted before fit-candidate selection, positive low-SNR points,
and above-unity transmission (negative measured attenuation). Nonpositive/undefined normalizations remain in
the raw panels and record table. Probe points with a changing companion are diagnostic measurements, not
additional comparable sweep points. Captured fit membership, coefficients, RMS, and acceptance are unchanged.

In [ ]:
def calibration_display_data(dataset, context):
    """Add display-only normalization for excluded sweep points; leave firmware support unchanged.

    The driver clips attenuation at zero and normalizes only classification='ok'. For plotting,
    positive unsaturated low-SNR readings and transmission above unity remain informative.
    Error bars include the same reference and bridge uncertainty, not independent fit weights.
    """
    # Support membership is determined from transmission, independent of the captured amplifier gain.
    frame = pd.DataFrame.from_records(dataset.derived())
    frame['display_db'] = np.nan
    frame['display_db_err'] = np.nan
    for physical in ('dac1', 'dac2'):
        meta = dataset._physical_meta(physical)
        rec = dataset.physical(physical).records
        if meta is None or not meta.get('reference_valid'):
            continue
        ref = dataset._record_by_index(rec, meta['reference_record'])
        if ref is None or ref.classification != 'ok' or ref.signal_mv <= 0 or ref.signal_err_mv <= 0:
            continue
        scales, variances = {0: 1.}, {0: 0.}
        for bridge in dataset._accepted_bridges(physical, rec):
            origin, dest = bridge['from_segment'], bridge['to_segment']
            if origin in scales:
                scales[dest] = scales[origin]*bridge['ratio']
                variances[dest] = variances[origin]+bridge['ratio_rel_var']
        for i, row in frame.loc[frame.physical.eq(physical)].iterrows():
            if row.event != 'point' or row.classification not in ('ok', 'below_snr') or row.segment not in scales:
                continue
            if row.signal_mv <= 0 or not np.isfinite(row.signal_err_mv) or row.signal_err_mv < 0:
                continue
            scaled = row.signal_mv/scales[row.segment]
            relative_var = (row.signal_err_mv/row.signal_mv)**2 + variances[row.segment]
            frame.loc[i, ['scaled_signal_mv', 'scaled_signal_err_mv']] = scaled, scaled*np.sqrt(relative_var)
            frame.loc[i, 'display_db'] = -10*np.log10(scaled/ref.signal_mv)
            frame.loc[i, 'display_db_err'] = 10/np.log(10)*np.sqrt(relative_var+(ref.signal_err_mv/ref.signal_mv)**2)
    return frame


def plot_calibration(dataset, context, physical, title=''):
    """Four panels with a flush, quarter-height residual panel. Pure offline display."""
    frame = calibration_display_data(dataset, context)
    p = frame.loc[frame.physical.eq(physical)].copy()
    if p.empty:
        print(f'{physical}: no retained records ({title}).')
        return None
    coeff = captured_coefficients(dataset, context).get(physical)
    fits = next((item['fits'] for item in dataset.meta if 'fits' in item), {})
    fit = fits.get(physical)
    fig = plt.figure(figsize=(12, 11))
    outer = fig.add_gridspec(3, 1, height_ratios=[1, 1, 2], hspace=.26)
    raw = fig.add_subplot(outer[0])
    scaled = fig.add_subplot(outer[1], sharex=raw)
    lower = outer[2].subgridspec(2, 1, height_ratios=[4, 1], hspace=0)
    curve_ax = fig.add_subplot(lower[0], sharex=raw)
    resid_ax = fig.add_subplot(lower[1], sharex=raw)
    axes = [raw, scaled, curve_ax, resid_ax]
    for classification, color in hspcb._ATTEN_CAL_CLASSIFICATION_COLORS.items():
        for event, marker in hspcb._ATTEN_CAL_EVENT_MARKERS.items():
            group = p.loc[p.classification.eq(classification) & p.event.eq(event)]
            if group.empty:
                continue
            raw.errorbar(group.sweep_mv, group.signal_mv, yerr=group.signal_err_mv.clip(lower=0),
                         fmt=marker, color=color, ms=3, elinewidth=.5, label=f'{event}: {classification}')
            # Variable-companion probes are diagnostics, not comparable sweep transmission.
            if event == 'point':
                good = group.loc[np.isfinite(group.scaled_signal_mv)]
                scaled.errorbar(good.sweep_mv, good.scaled_signal_mv, yerr=good.scaled_signal_err_mv,
                                fmt=marker, color=color, ms=3, elinewidth=.5)
    for role, indices in dataset._role_record_indices(physical).items():
        selected = p.loc[p.record.isin(indices)]
        for ax, col in ((raw, 'signal_mv'), (scaled, 'scaled_signal_mv')):
            ax.scatter(selected.sweep_mv, selected[col], marker=hspcb._ATTEN_CAL_ROLE_MARKERS[role],
                       facecolors='none', edgecolors=hspcb._ATTEN_CAL_ROLE_COLORS[role], s=65, label=role.replace('_', ' '))
    raw.set(ylabel='PD net (mV)', title='Raw dark-subtracted signal')
    scaled.set(ylabel='Scaled PD (mV)', title='Bridge-scaled signal')
    scaled.set_yscale('symlog', linthresh=.01)
    raw.legend(fontsize=7, ncol=3, loc='best')
    scaled.legend(fontsize=7, ncol=3, loc='best')
    measured = p.loc[np.isfinite(p.display_db)]
    if coeff is not None:
        grid = np.linspace(0, 3300, 3301)
        curve = hspcb._atten_db_from_coeff(coeff, grid)
        limit = float(fit.max_calibrated_db or 0)
        cross = np.flatnonzero(curve >= limit) if limit > 0 else np.array([], dtype=int)
        boundary = None
        if len(cross):
            k = int(cross[0])
            boundary = float(np.interp(limit, curve[max(0, k-1):k+1], grid[max(0, k-1):k+1]))
            grid = np.unique(np.r_[grid, boundary])
            curve = hspcb._atten_db_from_coeff(coeff, grid)
        inside = grid <= boundary if boundary is not None else np.zeros(len(grid), dtype=bool)
        curve_ax.plot(grid[inside], curve[inside], color='black', label='Captured firmware fit')
        curve_ax.plot(grid[~inside], curve[~inside], '--', color='black', alpha=.6, label='Rough continuation')
        if boundary is not None:
            curve_ax.axhline(limit, color='tab:purple', ls=':', lw=.8, label=f'Calibrated limit: {limit:g} dB')
            curve_ax.annotate(f'Rough region from {boundary:.1f} mV', (boundary, .98),
                              xycoords=('data', 'axes fraction'), xytext=(-5, -2), textcoords='offset points',
                              ha='right', va='top', color='tab:purple', fontsize=8)
            for ax in (curve_ax, resid_ax):
                ax.axvline(boundary, color='tab:purple', ls=':', lw=.8)
        in_range = measured.included & (measured.db.astype('float32') <= np.float32(limit))
        masks = [(in_range, 'o', 'tab:blue', 'Fit support in range'),
                 (measured.included & ~in_range, 'D', 'tab:orange', 'Fit support above limit'),
                 (~measured.included, 'x', '.5', 'Excluded')]
        tail = measured.loc[~measured.included & (measured.sweep_mv > measured.loc[measured.included, 'sweep_mv'].max())]
        if not tail.empty:
            for ax in (curve_ax, resid_ax):
                ax.axvspan(tail.sweep_mv.min(), 3300, color='.5', alpha=.10,
                           label='Excluded tail data' if ax is curve_ax else None)
        for mask, marker, color, label in masks:
            g = measured.loc[mask]
            if g.empty:
                continue
            curve_ax.errorbar(g.sweep_mv, g.display_db, yerr=g.display_db_err, fmt=marker,
                              color=color, ms=4, elinewidth=.6, label=label)
            residual = hspcb._atten_db_from_coeff(coeff, g.sweep_mv)-g.display_db
            resid_ax.errorbar(g.sweep_mv, residual, yerr=g.display_db_err,
                              fmt=marker, color=color, ms=3, elinewidth=.5)
        terms = max((i+1 for i, c in enumerate(coeff[4]) if c != 0), default=0)
        curve_ax.set_title(f'Captured fit: {terms} correction terms; accepted={fit.accepted}; firmware RMS={fit.rms_db:g} dB')
    else:
        curve_ax.errorbar(measured.sweep_mv, measured.display_db, yerr=measured.display_db_err,
                          fmt='x', color='.5', label='Excluded / no valid captured fit')
        curve_ax.set_title('No valid captured firmware fit; raw acquisition retained')
    for segment, group in p.loc[p.event.eq('point')].groupby('segment', sort=True):
        lo, hi = group.sweep_mv.min(), group.sweep_mv.max()
        if segment != 0:
            for ax in axes:
                ax.axvline(lo, color='.6', lw=.6, alpha=.35)
        curve_ax.text((lo+hi)/2, .03, f'segment {segment}', transform=curve_ax.get_xaxis_transform(),
                      ha='center', va='bottom', color='.5', fontsize=7)
    curve_ax.set_ylabel('Attenuation (dB)')
    curve_ax.legend(fontsize=7, loc='upper left')
    resid_ax.axhline(0, color='black', lw=.6)
    resid_ax.set(xlabel='DAC output (mV)', ylabel='Fit − data\n(dB)', xlim=(-30, 3330))
    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)
    # Avoid overlapping tick labels at the shared border without introducing a gap.
    resid_ax.yaxis.set_major_locator(plt.MaxNLocator(3, prune='both'))
    fig.suptitle(f'{physical} · {title}', fontsize=12)
    fig.subplots_adjust(top=.93, bottom=.07, left=.09, right=.98)
    return fig

In [ ]:
# Add real saved captures here; no connection or new calibration is required.
REPLAY_CALIBRATIONS = dict(CALIBRATION_FILES)
# REPLAY_CALIBRATIONS['1028y'] = TOOLS / 'atten_noise_data/cal_1028y_20260918T003155_207983Z.npz'
cal_fit_rows = []
for name, path in REPLAY_CALIBRATIONS.items():
    dataset, context = load_calibration(path)
    display(dataset.to_dataframe(), dataset.bridge_table())
    fits = next((item['fits'] for item in dataset.meta if 'fits' in item), {})
    for physical in ('dac1', 'dac2'):
        fit = fits.get(physical, hspcb.AttenuatorFitMetrics(valid=False))
        coeff = captured_coefficients(dataset, context).get(physical)
        curve = hspcb._atten_db_from_coeff(coeff, np.arange(3301)) if coeff else np.array([np.nan])
        cal_fit_rows.append(dict(laser=name, file=str(path), physical=physical,
            condition=parameter(context, 'condition', 'historical / unspecified'),
            acquisition_state=next((m.get('state') for m in dataset.meta if 'state' in m), 'unavailable'),
            retrieval_error=next((m.get('retrieval_error') for m in dataset.meta if 'retrieval_error' in m), ''),
            valid=fit.valid, accepted=fit.accepted, points=fit.points,
            max_calibrated_db=fit.max_calibrated_db, leakage_floor_db=fit.max_atten_db,
            rms_db=fit.rms_db, max_abs_db=fit.max_abs_db,
            monotonic=np.all(np.isfinite(curve)) and np.all(np.diff(curve) >= -1e-5)))
        plot_calibration(dataset, context, physical, title=f'{name} · {Path(path).name}')
cal_fit_table = pd.DataFrame(cal_fit_rows)
display(cal_fit_table)

In [ ]:
ACCEPT_CALIBRATION_FILES = {}           # e.g. {'1028y': Path('...reviewed capture.npz')}
PERSIST_ACCEPTED_CALIBRATIONS = False
if PERSIST_ACCEPTED_CALIBRATIONS:
    for name, path in ACCEPT_CALIBRATION_FILES.items():
        dataset, context = load_calibration(path)
        if parameter(context, 'laser') != name:
            raise ValueError('Capture laser identity does not match target.')
        fits = next(item['fits'] for item in dataset.meta if 'fits' in item)
        if any(m.get('retrieval_error') for m in dataset.meta) or any(m.get('state') != 'complete' for m in dataset.meta if 'state' in m):
            raise ValueError('Only a completed acquisition with no retrieval error can be persisted here.')
        if not all(fits[p].valid and fits[p].accepted for p in ('dac1', 'dac2')):
            raise ValueError('Both captured fits must be accepted before persisting this pair.')
        coefficients = {}
        for physical, c in captured_coefficients(dataset, context).items():
            coefficients[physical] = dict(fvoa_50pct_mv=c[0], slope_inv_fvoa_mv=c[1], max_atten_db=c[2],
                gain=c[3], correction_coeff=c[4], max_calibrated_db=c[5], rms_db=fits[physical].rms_db)
        pcb.atten_coeff(name, coefficients['dac1'], coefficients['dac2'], persist=True)
        save_tables(f'accepted_cal_{name}', source=np.array(str(path)), context=snapshot(pcb, name))

## Adjacent laser-current steps

Autocalibration is already complete. Query current settings again here; do not rerun it. The table starts with
minimum-autolevel and typical current for every laser. Typical means the presently positive current, otherwise
nominal, with its origin shown. Edit any row, choose subsets, or add a third operating point. Use previously
established FVOA settings that put the PD in range; keep both FVOAs fixed within an adjacent-current sequence.

Positive `laser(value=...)` spans threshold→nominal and quantizes to 0.1 mA; `value=0` means actual zero,
not threshold. A stored nonzero wavelength tune can alter the applied current/temperature: the notebook
records it and checks the applied setpoint. A flat 0.1 mA-resolution measured-current register does not prove
absence of finer or faster current noise. Temperature/readback cadence is separate from the optical stream.

In [ ]:
QUERY_CURRENT_SETUP = False
if QUERY_CURRENT_SETUP:
    current_rows = []
    for name in LASERS:
        settings, actual, drive = pcb.laser_settings(name), pcb.laser_status(name), pcb.atten(name)
        typical = actual.i_mA if actual.i_mA is not None and actual.i_mA > 0 else settings.nominal_current_ma
        for label, current, origin in [('minimum', settings.min_autolevel_current_ma, 'queried minimum autolevel'),
                ('typical', typical, 'present positive current' if actual.i_mA is not None and actual.i_mA > 0 else 'nominal fallback')]:
            current_rows.append(dict(laser=name, point=label, current_ma=current, origin=origin,
                threshold_ma=settings.threshold_current_ma, nominal_ma=settings.nominal_current_ma,
                measured_ma=actual.curr_meas_ma, tune_nm=settings.tune_nm,
                dac1_mv=drive.v1_mv, dac2_mv=drive.v2_mv, seconds=10.0))
    current_points = pd.DataFrame(current_rows)
    display(current_points)
# Edit in place; a third point is just another row. Durations are independent, not a total runtime budget.
# current_points.loc[0, ['current_ma', 'seconds']] = [14.6, 20.]
# current_points.loc[len(current_points)] = {**current_points.iloc[0].to_dict(), 'point': 'third', 'current_ma': 20.0}

In [ ]:
def current_fraction(settings, requested_ma):
    """Translate one reachable Maiman setpoint through the existing fractional command API."""
    if requested_ma == 0:
        return 0.
    low = math.ceil((settings.threshold_current_ma + 1e-9)*10)/10
    high = math.floor((settings.nominal_current_ma + 1e-9)*10)/10
    if not low <= requested_ma <= high or not np.isclose(requested_ma*10, round(requested_ma*10), atol=1e-7):
        raise ValueError(f'{requested_ma} mA is not a reachable positive 0.1 mA setpoint in [{low}, {high}].')
    return (requested_ma-settings.threshold_current_ma)/(settings.nominal_current_ma-settings.threshold_current_ma)

In [ ]:
RUN_CURRENT_STEPS = False
CURRENT_REPEATS = 2
if RUN_CURRENT_STEPS:
    for point in current_points.itertuples(index=False):
        settings = pcb.laser_settings(point.laser)
        center = round(point.current_ma*10)/10
        high = math.floor((settings.nominal_current_ma+1e-9)*10)/10
        neighbor = round((center+.1 if center+.1 <= high else center-.1)*10)/10
        for repeat in range(CURRENT_REPEATS):
            for step, requested in enumerate((center, neighbor, center)):
                fraction = current_fraction(settings, requested)
                monitor = open_stream(pcb, point.seconds, laser=point.laser)
                try:
                    for name in LASERS:
                        if name != point.laser:
                            pcb.laser(name, value=0, autooff_s=0)
                    pcb.atten(point.laser, value1_mv=point.dac1_mv, value2_mv=point.dac2_mv)
                    before_ms = time.time_ns()//1_000_000
                    pcb.laser(point.laser, value=fraction, autooff_s=0)
                    after_ms = time.time_ns()//1_000_000
                    actual = pcb.laser_status(point.laser)
                    context = snapshot(pcb, point.laser)
                    collect_trace(pcb, monitor, point.laser, point.seconds, 'current_steps', context=context,
                        extra=dict(point=point.point, repeat=repeat, step=step, requested_current_ma=requested,
                            applied_current_ma=actual.i_mA, measured_current_ma=actual.curr_meas_ma,
                            command_start_ms=before_ms, command_end_ms=after_ms,
                            current_match=np.isclose(actual.i_mA, requested, atol=.049) if actual.i_mA is not None else False))
                finally:
                    monitor.stop()
        pcb.laser(point.laser, value=0, autooff_s=0)

## Independent fixed-current holds and discontinuous revisits

This section stands on its own. Query the selected laser, settings, coefficients, dark, routing, and FVOAs;
edit the visible operating point, then capture. A 5 s, 200 s, or longer hold uses the same code with capacity
scaled to the requested duration. Warmup is observed in the trace, not guaranteed by a fixed delay.

The separate zero-current cell clears the auto-off deadline and keeps an already enabled driver ready. Return
half an hour later and run another hold: archives retain absolute board and host times, without joining the gap.
The explicit shutdown cell stops the driver and applies its TEC-off policy. No routine static-attenuator bypass
experiment is required; previous quiet bypass data are supporting evidence, not proof of the present noise floor.

In [ ]:
QUERY_HOLD_SETUP = False
HOLD_LASER = LASERS[0]
HOLD_SECONDS = 30.0
if QUERY_HOLD_SETUP:
    display(snapshot(pcb, HOLD_LASER))
    settings = pcb.laser_settings(HOLD_LASER)
    actual = pcb.laser_status(HOLD_LASER)
    drive = pcb.atten(HOLD_LASER)
    HOLD_CURRENT_MA = actual.i_mA if actual.i_mA is not None and actual.i_mA > 0 else settings.min_autolevel_current_ma
    HOLD_DAC_MV = (drive.v1_mv, drive.v2_mv)
    display(pd.DataFrame([dict(laser=HOLD_LASER, current_ma=HOLD_CURRENT_MA,
        dac1_mv=HOLD_DAC_MV[0], dac2_mv=HOLD_DAC_MV[1], seconds=HOLD_SECONDS)]))
# Edit HOLD_CURRENT_MA / HOLD_DAC_MV after querying; changing duration never reruns autocalibration.

In [ ]:
RUN_HOLD = False
if RUN_HOLD:
    settings = pcb.laser_settings(HOLD_LASER)
    fraction = current_fraction(settings, HOLD_CURRENT_MA)
    monitor = open_stream(pcb, HOLD_SECONDS, laser=HOLD_LASER)
    try:
        for name in LASERS:
            if name != HOLD_LASER:
                pcb.laser(name, value=0, autooff_s=0)
        pcb.atten(HOLD_LASER, value1_mv=HOLD_DAC_MV[0], value2_mv=HOLD_DAC_MV[1])
        command_start_ms = time.time_ns()//1_000_000
        pcb.laser(HOLD_LASER, value=fraction, autooff_s=0)
        command_end_ms = time.time_ns()//1_000_000
        context = snapshot(pcb, HOLD_LASER)
        hold_file = collect_trace(pcb, monitor, HOLD_LASER, HOLD_SECONDS, 'hold', context=context,
            extra=dict(requested_current_ma=HOLD_CURRENT_MA, command_start_ms=command_start_ms, command_end_ms=command_end_ms))
    finally:
        monitor.stop()

In [ ]:
ZERO_AND_REMAIN_ENABLED = False
if ZERO_AND_REMAIN_ENABLED:
    pcb.stop_throughput(CHANNEL)         # Manual stream ownership; no inherited autolevel owner.
    pcb.laser(HOLD_LASER, value=0, autooff_s=0)
    save_tables(f'idle_zero_{HOLD_LASER}', context=snapshot(pcb, HOLD_LASER))

In [ ]:
FULL_SHUTDOWN_SELECTED = False
if FULL_SHUTDOWN_SELECTED:
    try:
        pcb.stop_throughput(CHANNEL)
    finally:
        pcb.laser(HOLD_LASER, stop=True)
    save_tables(f'shutdown_{HOLD_LASER}', context=snapshot(pcb, HOLD_LASER))

In [ ]:
RUN_AUTOOFF_EXERCISE = False
AUTOOFF_SECONDS = 5
if RUN_AUTOOFF_EXERCISE:
    pcb.stop_throughput(CHANNEL)
    settings = pcb.laser_settings(HOLD_LASER)
    pcb.atten(HOLD_LASER, value1_mv=HOLD_DAC_MV[0], value2_mv=HOLD_DAC_MV[1])
    pcb.laser(HOLD_LASER, value=current_fraction(settings, HOLD_CURRENT_MA), autooff_s=AUTOOFF_SECONDS)
    # Plain zero keeps this deadline, contrasting with the intentional indefinite idle cell above.
    pcb.laser(HOLD_LASER, value=0)
    observations = []
    try:
        end = time.monotonic()+AUTOOFF_SECONDS+2
        while time.monotonic() < end:
            observations.append(parameter_table(dict(host_ms=time.time_ns()//1_000_000,
                laser=pcb.laser(HOLD_LASER), engineering=pcb.laser_status(HOLD_LASER))))
            time.sleep(.5)
    finally:
        save_tables(f'autooff_{HOLD_LASER}', observations=pd.concat(observations, keys=range(len(observations)))
                    .reset_index(level=0).rename(columns={'level_0': 'observation'}) if observations else parameter_table({}))
        pcb.laser(HOLD_LASER, stop=True)

## Repeated levels, attenuation redistribution, and tail checks

Query each laser's existing settings before building this experiment. The default row order compares
25+35 → 35+25 → 25+25 → 25+35 dB at one current, then scans a physical FVOA through its calibrated boundary.
Edit operating points to suit the assembly and PD range. Raw records keep settling, overrange and dark data;
analysis selects stationary usable intervals explicitly. Calibrated boundaries come from queried coefficients,
not an assumed 55 dB for every device. Commands beyond the boundary probe a rough model and are labeled so.

In [ ]:
QUERY_NOISE_SETUP = False
if QUERY_NOISE_SETUP:
    noise_rows = []
    for name in LASERS:
        settings, actual, coefficients = pcb.laser_settings(name), pcb.laser_status(name), pcb.atten_coeff(name)
        current = actual.i_mA if actual.i_mA is not None and actual.i_mA > 0 else settings.min_autolevel_current_ma
        for label, a, b in [('redistribute_A',25.,35.), ('redistribute_B',35.,25.), ('brighter',25.,25.), ('redistribute_A_repeat',25.,35.)]:
            noise_rows.append(dict(laser=name, point=label, current_ma=current, dac1_db=a, dac2_db=b, seconds=30.))
        for physical in ('dac1', 'dac2'):
            limit = getattr(coefficients, physical).max_calibrated_db
            for offset in (-2.5, 0, 2.5, 5.):
                value = max(0., limit+offset)
                noise_rows.append(dict(laser=name, point=f'{physical}_limit{offset:+g}', current_ma=current,
                    dac1_db=value if physical == 'dac1' else 0., dac2_db=value if physical == 'dac2' else 0., seconds=30.))
    noise_points = pd.DataFrame(noise_rows)
    display(noise_points)

In [ ]:
RUN_NOISE_POINTS = False
if RUN_NOISE_POINTS:
    for point in noise_points.itertuples(index=False):
        settings = pcb.laser_settings(point.laser)
        fraction = current_fraction(settings, round(point.current_ma*10)/10)
        monitor = open_stream(pcb, point.seconds, laser=point.laser)
        try:
            for name in LASERS:
                if name != point.laser:
                    pcb.laser(name, value=0, autooff_s=0)
            command_start_ms = time.time_ns()//1_000_000
            applied = pcb.atten(point.laser, value1_db=point.dac1_db, value2_db=point.dac2_db)
            pcb.laser(point.laser, value=fraction, autooff_s=0)
            command_end_ms = time.time_ns()//1_000_000
            context = snapshot(pcb, point.laser)
            collect_trace(pcb, monitor, point.laser, point.seconds, 'noise', context=context,
                extra=dict(point=point.point, requested_dac1_db=point.dac1_db, requested_dac2_db=point.dac2_db,
                    applied_dac1_mv=applied.v1_mv, applied_dac2_mv=applied.v2_mv,
                    command_start_ms=command_start_ms, command_end_ms=command_end_ms))
        finally:
            monitor.stop()
    for name in LASERS:
        pcb.laser(name, value=0, autooff_s=0)

## Throughput, autolevel, and response

Run the installed policy and label the build. The editable table includes all three lasers and both outputs;
select rows matching the physical patch, then repeat after repatching. The collector/dashboard is the existing
driver implementation. Autolevel changes current/attenuation and owns laser shutdown. Manual commands disable
its adjustment loop but retain that ownership, which is why later manual sections stop it **before** source setup.

The 50 ms publication cadence is one fresh ADC conversion, not a continuous 50 ms integration. Missing records
are gaps. The 500 ms PD diagnostic window and configurable calibration averages are separate measurements.
The firmware's 400–1600 mV useful band, movements, and settling lag remain visible. A throughput peak is a
loose loopback reference; there is no unity target. Values depend on measured route losses and wavelength
response assumptions. An overrange reading is a nominal lower bound, not a precise calibration point.

In [ ]:
throughput_points = pd.DataFrame([dict(laser=name, output=f'{CHANNEL}_{out}', fiber=FIBER,
    initial_level=.5, seconds=60., patch_label=f'loopback_{out}_{FIBER}') for name in LASERS for out in ('ao', 'fei')])
display(throughput_points)
RUN_THROUGHPUT = False
THROUGHPUT_ROW = 0
if RUN_THROUGHPUT:
    point = throughput_points.iloc[THROUGHPUT_ROW]
    context = pd.concat([snapshot(pcb, point.laser, output=point.output, fiber=point.fiber), parameter_table({'run': point.to_dict()})])
    monitor = open_stream(pcb, point.seconds, laser=point.laser, fiber=point.fiber, output=point.output,
                          autolevel=True, initial_level=point.initial_level)
    throughput_file = collect_trace(pcb, monitor, point.laser, point.seconds, 'throughput', context=context,
                                   extra=dict(point=point.patch_label))

In [ ]:
# Optional live exploration; execute this cell, then the save/stop cell when finished.
# Enable an interactive backend with %matplotlib widget in a separate cell if desired.
START_LIVE_DASHBOARD = False
LIVE_LASER, LIVE_CAPACITY_SECONDS = LASERS[0], 600.
if START_LIVE_DASHBOARD:
    if 'tp_figure' in globals():
        plt.close(tp_figure)
    live_context = snapshot(pcb, LIVE_LASER)
    monitor = open_stream(pcb, LIVE_CAPACITY_SECONDS, laser=LIVE_LASER, autolevel=True, initial_level=.5)
    tp_figure, tp_animation = monitor.plot_live(channel=CHANNEL, max_points=600)
    display(tp_figure)
# tp_animation.pause()/resume() affect only display. Hardware and collection keep running.

In [ ]:
SAVE_AND_STOP_LIVE = False
if SAVE_AND_STOP_LIVE:
    try:
        monitor.stop()
    finally:
        samples = monitor.to_dataframe()
        samples['source_laser'], samples['experiment'] = LIVE_LASER, 'throughput_live'
        samples['condition'], samples['build_label'] = CONDITION, BUILD_LABEL
        path = save_tables(f'throughput_live_{LIVE_LASER}', samples=samples, context=live_context,
            outcome=parameter_table(dict(capacity=monitor.max_samples, capacity_reached=len(samples)>=monitor.max_samples)))
        CAPTURE_FILES.append(path)

## Offline analysis and cross-laser / environmental comparison

Select archives explicitly to compare room and chamber data, different laser currents, attenuation distributions,
and both firmware builds. Every file/interval remains separate. Drift and RMS answer different questions;
plots retain the full trace as well as stationary-segment statistics. The averaging plot compares measured
block-mean scatter with an independent-sample prediction; adjacent-block correlation and Allan deviation help
identify when that prediction fails. Shared calibration uncertainty never shrinks as 1/√N here.

The PSD uses contiguous samples and their median interval; timing jitter is reported. It is not a frequency
response correction for arbitrary irregular sampling. Temporal PD scatter is not `tp_err`. The latter includes
source/curve calibration terms and a static independent-drive electrical-noise assumption (10 mV per FVOA).
Do not equate residual fit RMS with electrical temporal noise or assume all excess noise comes from FVOAs.

In [ ]:
def stationary_segments(frame, settle_s=1.0, minimum_s=2.0):
    """Keep separate contiguous fixed-setting intervals; do not close gaps by dropping bad rows."""
    if len(frame) < 3:
        return []
    frame = frame.reset_index(drop=True)
    dt = frame.t_ms.diff()/1000
    nominal = dt[dt > 0].median()
    valid = np.isfinite(frame.pd_net_mv) & np.isfinite(frame.pd_mv) & frame.pd_mv.lt(hspcb.PD_ADC_USABLE_MV)
    valid &= ~frame.autolevel.astype(bool)
    boundary = (dt <= 0) | (dt > 1.5*nominal) | ~valid | ~valid.shift(fill_value=False)
    for column in ('channel', 'laser', 'experiment', 'point', 'step', 'step_index', 'selection'):
        if column in frame:
            labels = frame[column].astype('string').fillna('')
            boundary |= labels.ne(labels.shift(fill_value=''))
    # 0.1 mA steps are the experiment: never merge them using the old 0.2 mA tolerance.
    for column, tolerance in (('laser_current_ma', .049), ('atten_db', .005)):
        boundary |= frame[column].diff().abs().gt(tolerance)
    parts = []
    for _, group in frame.groupby(boundary.cumsum(), sort=False):
        if not valid.loc[group.index].all():
            continue
        group = group.loc[(group.t_ms-group.t_ms.iloc[0])/1000 >= settle_s]
        if len(group) >= 3 and (group.t_ms.iloc[-1]-group.t_ms.iloc[0])/1000 >= minimum_s:
            parts.append(group.reset_index(drop=True))
    return parts


def noise_statistics(part):
    """Empirical scatter, drift, spectrum, ACF and averaging; shared calibration errors are separate."""
    y = part.pd_net_mv.to_numpy(float)
    t = (part.t_ms.to_numpy(float)-float(part.t_ms.iloc[0]))/1000
    dt, rms = np.diff(t), np.std(y, ddof=1)
    fs = 1/np.median(dt)
    centered = y-y.mean()
    acf = signal.correlate(centered, centered, mode='full', method='fft')[len(y)-1:]/np.arange(len(y), 0, -1)
    acf = acf/acf[0] if acf[0] > 0 else np.full_like(acf, np.nan)
    f, psd = signal.welch(y, fs=fs, nperseg=min(1024, max(3, len(y)//4)), detrend='constant')
    blocks = []
    for count in np.unique(np.maximum(1, np.round(np.array([.05,.1,.25,.5,1,2,5,10,20,30,60])*fs).astype(int))):
        n = len(y)//count
        if n < 4:
            continue
        means = y[:n*count].reshape(n, count).mean(axis=1)
        blocks.append(dict(tau_s=count/fs, blocks=n, rms_mean_mv=np.std(means, ddof=1),
            iid_rms_mean_mv=rms/np.sqrt(count), allan_mv=np.sqrt(.5*np.mean(np.diff(means)**2)),
            adjacent_block_corr=np.corrcoef(means[:-1], means[1:])[0,1] if np.std(means)>0 else np.nan))
    metrics = dict(samples=len(y), duration_s=t[-1], mean_mv=y.mean(), rms_mv=rms,
        detrended_rms_mv=np.std(signal.detrend(y), ddof=1), drift_mv_per_s=np.polyfit(t, y, 1)[0],
        relative_rms=rms/abs(y.mean()) if abs(y.mean()) > 3*rms else np.nan,
        adjacent_corr=acf[1], sample_hz=fs, max_gap_ms=1000*max(dt), timing_jitter_ms=1000*np.std(dt))
    return metrics, pd.DataFrame(dict(f_hz=f, psd_mv2_per_hz=psd)), pd.DataFrame(blocks), acf

In [ ]:
ANALYSIS_FILES = list(CAPTURE_FILES)      # Add archived room / chamber / other-build files here.
SETTLE_S, MIN_STATIONARY_S = 1., 2.
summary_rows, spectrum_rows, averaging_rows, audit_rows, throughput_rows = [], [], [], [], []
for path in ANALYSIS_FILES:
    saved = load_tables(path)
    if 'samples' not in saved or saved['samples'].empty:
        continue
    data, context = saved['samples'], saved['context']
    identity = dict(file=str(path), laser=parameter(context, 'laser'), condition=parameter(context, 'condition'),
                    build=parameter(context, 'build_label'), ambient_c=parameter(context, 'status.amb_c'))
    dt = data.t_ms.diff()
    audit_rows.append(dict(**identity, rows=len(data), median_dt_ms=dt[dt>0].median(),
        gaps_over_75ms=int((dt>75).sum()), duplicate_or_reversed=int((dt<=0).sum()),
        overrange=int((data.pd_mv>=hspcb.PD_ADC_USABLE_MV).sum()), autolevel_rows=int(data.autolevel.sum())))
    t = (data.t_ms-data.t_ms.iloc[0])/1000
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(11,7), layout='constrained')
    axes[0].plot(t, data.pd_net_mv, lw=.6); axes[0].set_ylabel('PD net (mV)')
    axes[1].plot(t, data.laser_current_ma); axes[1].set_ylabel('Current set (mA)')
    axes[2].plot(t, data.atten_db); axes[2].set(ylabel='Pair attenuation (dB)', xlabel='Seconds from first retained sample')
    fig.suptitle(f'{Path(path).name} · {identity["condition"]} · {identity["build"]}')
    if 'telemetry' in saved and not saved['telemetry'].empty:
        telemetry = saved['telemetry']
        tt = ((telemetry.host_start_ms+telemetry.host_end_ms)/2-telemetry.host_start_ms.iloc[0])/1000
        tele_fig, tele_axes = plt.subplots(2,1,sharex=True,figsize=(11,5),layout='constrained')
        tele_axes[0].plot(tt,telemetry.i_mA,'.-',label='Confirmed setpoint')
        tele_axes[0].plot(tt,telemetry.curr_meas_ma,'.-',label='Measured (0.1 mA resolution)')
        tele_axes[0].set_ylabel('Current (mA)');tele_axes[0].legend(fontsize=8)
        for column in ('tec_temp_c','pcb_temp_c','ambient_c'):
            tele_axes[1].plot(tt,telemetry[column],'.-',label=column)
        tele_axes[1].set(xlabel='Host seconds from first engineering read',ylabel='Temperature (°C)')
        tele_axes[1].legend(fontsize=8)
        tele_fig.suptitle(Path(path).name+' · slow engineering telemetry')
    for interval, part in enumerate(stationary_segments(data, SETTLE_S, MIN_STATIONARY_S)):
        stats, psd, blocks, acf = noise_statistics(part)
        tags = dict(**identity, interval=interval, point=part.point.iloc[0] if 'point' in part else part.experiment.iloc[0])
        summary_rows.append(dict(**tags, start_t_ms=int(part.t_ms.iloc[0]),
            current_ma=part.laser_current_ma.median(), atten_db=part.atten_db.median(),
            dac1_mv=parameter(context,'atten.v1_mv'), dac2_mv=parameter(context,'atten.v2_mv'),
            effective_gain_v_per_a=parameter(context,'pd_settings.transimpedance_v_per_a'),
            active_dark_rms_mv=parameter(context,'dark.dark.rms_mv'),
            **stats))
        spectrum_rows.append(psd.assign(**tags))
        averaging_rows.append(blocks.assign(**tags))
    valid = data.loc[np.isfinite(data.tp) & (data.pd_mv < hspcb.PD_ADC_USABLE_MV)]
    if not valid.empty:
        throughput_rows.append(dict(**identity, count=len(valid), median_tp=valid.tp.median(),
            p95_tp=valid.tp.quantile(.95), peak_tp=valid.tp.max(), temporal_tp_rms=valid.tp.std(),
            median_model_uncertainty=valid.tp_err.median(), median_pd_uncertainty=valid.tp_pd_err.median(),
            pd_route_tx=valid.pd_route_tx.median(), launch_route_tx=valid.laser_route_tx.median(),
            recompute_max_abs_error=(valid.pd_power_nw/valid.delivered_power_nw-valid.tp).abs().max()))
noise_summary = pd.DataFrame(summary_rows)
spectra = pd.concat(spectrum_rows, ignore_index=True) if spectrum_rows else pd.DataFrame()
averaging = pd.concat(averaging_rows, ignore_index=True) if averaging_rows else pd.DataFrame()
throughput_summary = pd.DataFrame(throughput_rows)
display(pd.DataFrame(audit_rows), noise_summary, throughput_summary)
if not noise_summary.empty:
    display(noise_summary.pivot_table(index=['laser','point','current_ma'], columns=['condition','build'],
        values=['mean_mv','rms_mv','drift_mv_per_s'], aggfunc='mean'))
    fig, axes = plt.subplots(1, 2, figsize=(12,4), layout='constrained')
    for (file, interval), group in spectra.groupby(['file','interval'], sort=False):
        axes[0].loglog(group.f_hz, np.sqrt(group.psd_mv2_per_hz), label=f'{Path(file).stem}:{interval}')
    for (file, interval), group in averaging.groupby(['file','interval'], sort=False):
        line, = axes[1].loglog(group.tau_s, group.rms_mean_mv, '.-', label=f'{Path(file).stem}:{interval}')
        axes[1].loglog(group.tau_s, group.iid_rms_mean_mv, ':', color=line.get_color(), alpha=.6)
    axes[0].set(xlabel='Frequency (Hz)', ylabel='PD amplitude density (mV/√Hz)')
    axes[1].set(xlabel='Block duration (s)', ylabel='Block-mean RMS (mV)')
    for ax in axes:
        ax.legend(fontsize=6)

### Shot-noise scale and normalization assumptions

For mean ADC-input signal V and effective transimpedance G (including the divider), the elementary photocurrent
shot-noise scale is √(2qVGB). Here B is an **equivalent noise bandwidth**, not the detector's quoted −3 dB
bandwidth or the telemetry Nyquist frequency. Enter measured/justified bandwidths below; scenarios remain
scenarios. A nearby measured dark captures detector/electronics noise already present. Excess after quadrature
subtraction is descriptive: drift, laser, FVOA, detector and acquisition effects are not separated by this number.
Per-laser responsivity has no separate current command API; inspect the channel setting/wavelength assumptions
and keep externally measured values in the tables instead of silently rewriting calibration during analysis.

In [ ]:
ENBW_SCENARIOS_HZ = [5., 10., 20.]      # Illustrative scenarios, not measured ADC/detector response.
# Optional per-file adjacent dark overrides; otherwise use that capture's active dark RMS.
DARK_RMS_OVERRIDES = {}
shot_rows = []
for row in noise_summary.itertuples(index=False):
    for bandwidth in ENBW_SCENARIOS_HZ:
        dark_rms = DARK_RMS_OVERRIDES.get(row.file, row.active_dark_rms_mv)
        shot_mv = 1000*np.sqrt(2*1.602176634e-19*max(0,row.mean_mv/1000)*row.effective_gain_v_per_a*bandwidth)
        excess_variance = row.rms_mv**2-dark_rms**2-shot_mv**2
        shot_rows.append(dict(file=row.file, laser=row.laser, interval=row.interval, condition=row.condition,
            enbw_scenario_hz=bandwidth, measured_rms_mv=row.rms_mv, dark_rms_mv=dark_rms, shot_rms_mv=shot_mv,
            excess_variance_mv2=excess_variance, excess_rms_mv=np.sqrt(max(0,excess_variance)) if np.isfinite(excess_variance) else np.nan))
shot_table = pd.DataFrame(shot_rows)
display(shot_table)
SAVE_ANALYSIS = False
if SAVE_ANALYSIS:
    save_tables('analysis', noise_summary=noise_summary, throughput_summary=throughput_summary,
        spectra=spectra, averaging=averaging, shot_scenarios=shot_table)

### Commanded response and repeatability

The current-step and noise captures retain startup and command transitions. Select one contiguous interval
around a transition and inspect its trace before quoting response times. Below, crossing times are relative
to the observed optical 10% crossing; command host-time brackets are saved but are not an oscilloscope trigger.
At 20 Hz, sub-sample response times are unresolved. The separate scope notebook measures simultaneous drive
and PD steps when the firmware stream is too slow.

In [ ]:
RESPONSE_FILE = None
RESPONSE_WINDOW_S = (0., 10.)
BASELINE_WINDOW_S = (0., 1.)
FINAL_WINDOW_S = (8., 10.)
if RESPONSE_FILE is not None:
    response = load_tables(RESPONSE_FILE)['samples']
    rt = (response.t_ms-response.t_ms.iloc[0])/1000
    select = rt.between(*RESPONSE_WINDOW_S)
    baseline = response.loc[rt.between(*BASELINE_WINDOW_S),'pd_net_mv'].median()
    final = response.loc[rt.between(*FINAL_WINDOW_S),'pd_net_mv'].median()
    relative = (response.pd_net_mv-baseline)/(final-baseline)
    crossing = {}
    for fraction in (.1, .5, .9):
        hits = rt.loc[select & relative.ge(fraction)]
        crossing[fraction] = hits.iloc[0] if len(hits) else np.nan
    display(pd.DataFrame([dict(baseline_mv=baseline, final_mv=final,
        t10_s=crossing[.1], t50_s=crossing[.5], t90_s=crossing[.9],
        rise_10_90_s=crossing[.9]-crossing[.1], sample_interval_s=rt.diff().median())]))
    plt.plot(rt.loc[select], relative.loc[select], '.-')
    plt.xlabel('Seconds from first retained sample'); plt.ylabel('Fraction of observed step')

## Commissioning record

Archive the selected measurements and annotate conclusions in this table: what the optical routing matrix
established, assembled losses adopted, fit acceptance/range, current-step repeatability, warmup/drift/noise,
response limits, and throughput references. Compare all three lasers and both environments, with the active
calibration and build attached. Leave unmeasured items explicitly unmeasured; no arbitrary acceptance limits
are supplied by the notebook. Later HISPEC acquisition code or a future driver can adopt procedures after
bench validation without importing this notebook as a hidden dependency.

In [ ]:
commissioning_record = pd.DataFrame(columns=['condition','build','laser','function','capture','observation','adopted_value','units','remaining_question'])
SAVE_COMMISSIONING_RECORD = False
if SAVE_COMMISSIONING_RECORD:
    save_tables('commissioning_record', conclusions=commissioning_record)
display(commissioning_record)

## Offline verification of this notebook

Notebook syntax/schema and offline execution were checked without a hardware connection. The successful
`cal_1028y_20260918T003155_207983Z`, rejected-fit `cal_1028y_20260918T230245_185968Z`, and empty
`cal_1028y_20260918T232236_170469Z` captures were replayed and round-tripped through the table archives.
Combined figures were rendered and inspected. Existing noise records were analyzed; synthetic current changes
and discontinuous holds checked interval separation. A recording PCB double checked public-driver call
signatures, retro routing, current sequences, idle/shutdown, and archive-before-cancellation ordering.
These checks establish notebook behavior, not bench calibration accuracy. Hardware, environmental-chamber
and both-build measurements remain to be made with the acquisition cells.